# EfficientNet V2 + YOLOv8 Segmentacja

**Pipeline:**
1. YOLOv8-seg wycina maski świń z wszystkich obrazów (train2 + test) i zapisuje na dysk
2. EfficientNet V2 trenuje na wyczyszczonych cropach — pełna prędkość
3. TTA przy inferencji (tylko brightness, bez flipów)
4. Checkpointy po każdej epoce

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 600)

In [ ]:
!kaggle competitions download -c multi-view-pig-posture-recognition

In [ ]:
!unzip -q multi-view-pig-posture-recognition.zip -d data

In [ ]:
!ls data/multiview_pig_posture_recognition

## Instalacja YOLOv8

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

## Importy

In [ ]:
import os, ast, numpy as np, pandas as pd
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from ultralytics import YOLO

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

## Konfiguracja

In [ ]:
CLASS_NAMES = {
    0: "Lateral_lying_left",
    1: "Lateral_lying_right",
    2: "Sitting",
    3: "Standing",
    4: "Sternal_lying",
}

BASE_DIR   = "data/multiview_pig_posture_recognition"
MASK_DIR   = "pig_masks"   # tu zapisujemy wycięte cropów z maskami
BATCH_SIZE = 32
EPOCHS     = 6
IMG_SIZE   = 224

os.makedirs(MASK_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Wczytanie danych

In [ ]:
train2 = pd.read_csv(os.path.join(BASE_DIR, "train2.csv"))
test   = pd.read_csv(os.path.join(BASE_DIR, "test.csv"))

train2["source"] = "train2"
test["source"]   = "test"

for df in [train2, test]:
    df["bbox_parsed"] = df["bbox"].apply(ast.literal_eval)

train2["class_name"] = train2["class_id"].map(CLASS_NAMES)
train_df = train2.reset_index(drop=True)

print("Train2:", len(train_df))
print("Test:",   len(test))
print()
print(train_df["class_name"].value_counts())

## Funkcje pomocnicze

In [ ]:
def load_image(image_id, source):
    folder_map = {
        "train2": "train2_images",
        "test":   "test_images",
    }
    path = os.path.join(BASE_DIR, folder_map[source], image_id)
    return Image.open(path).convert("RGB")


def crop_square(image_pil, bbox_parsed, padding=0.12):
    """Standardowy crop kwadratowy — fallback gdy YOLO nie znajdzie świni."""
    img_w, img_h = image_pil.size
    x, y, w, h = map(float, bbox_parsed)
    pad_x, pad_y = w * padding, h * padding
    x1, y1 = x - pad_x, y - pad_y
    x2, y2 = x + w + pad_x, y + h + pad_y
    side = max(x2 - x1, y2 - y1)
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    x1 = max(0, int(round(cx - side/2)))
    y1 = max(0, int(round(cy - side/2)))
    x2 = min(img_w, int(round(cx + side/2)))
    y2 = min(img_h, int(round(cy + side/2)))
    return image_pil.crop((x1, y1, x2, y2))


def mask_path_for_row(row):
    """Ścieżka gdzie zapisujemy gotowy crop z maską dla danej instancji."""
    safe_id = row["row_id"].replace("/", "_").replace("\\", "_")
    return os.path.join(MASK_DIR, f"{safe_id}.png")

## Inicjalizacja YOLOv8-seg

Używamy `yolov8m-seg` — wariant medium, dobry balans prędkości i dokładności.
Model jest pretrenowany na COCO który zawiera klasę `pig` (klasa nr 17).

In [ ]:
# Pobierz YOLOv8m-seg (medium — dobry balans speed/accuracy)
yolo_model = YOLO("yolov8m-seg.pt")

# Sprawdź czy pig jest w klasach COCO
PIG_CLASS_ID = 17  # COCO class: "horse"=17... sprawdzamy
coco_names = yolo_model.names
print("Klasy COCO (pierwsze 20):")
for k, v in list(coco_names.items())[:20]:
    print(f"  {k}: {v}")

# Znajdź pig
pig_ids = [k for k, v in coco_names.items() if "pig" in v.lower() or "swine" in v.lower() or "hog" in v.lower()]
print("\nZnalezione klasy świni:", pig_ids, [coco_names[i] for i in pig_ids])

In [ ]:
# COCO klasa 17 = "horse", klasa 19 = "cow", klasa 20 = "sheep"
# Świnia to nie ma osobnej klasy w COCO — używamy detect na wszystko
# i wybieramy detekcję która pokrywa się z naszym bounding boxem

# Alternatywnie — używamy YOLO tylko do segmentacji obiektu w bbox
# bez filtrowania po klasie (bo i tak mamy bbox z datasetu)
PIG_CLASSES = None  # None = wszystkie klasy, wybieramy po IoU z bbox
print("YOLO gotowy — będziemy wybierać detekcję po IoU z bounding boxem z datasetu")

## Krok 1 — Generowanie masek YOLOv8 dla całego train2 + test

Dla każdej instancji (wiersz w CSV):
1. Wczytaj obraz
2. Uruchom YOLOv8-seg na całym obrazie
3. Znajdź detekcję której bbox najlepiej pokrywa się z naszym bbox
4. Zastosuj maskę (przyciemnij tło)
5. Zapisz crop na dysk

Fallback: jeśli YOLO nic nie znajdzie → standardowy crop bez maski.

In [ ]:
def compute_iou(box1, box2):
    """IoU między dwoma bboxami w formacie xyxy."""
    x1 = max(box1[0], box2[0]); y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2]); y2 = min(box1[3], box2[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    if inter == 0:
        return 0.0
    a1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    a2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    return inter / (a1 + a2 - inter)


def generate_masked_crop(row, yolo_model, padding=0.12, bg_darken=0.25, iou_threshold=0.2):
    """
    Generuje crop świni z wytłumionym tłem używając YOLOv8-seg.
    
    - Uruchamia YOLO na całym obrazie
    - Wybiera detekcję z najwyższym IoU z bbox z datasetu  
    - Przyciemnia tło do bg_darken (0.25 = 25% jasności)
    - Fallback na standardowy crop jeśli IoU < iou_threshold
    
    Zwraca PIL Image gotowy do zapisu.
    """
    image_pil = load_image(row["image_id"], row["source"])
    img_np    = np.array(image_pil)
    img_h, img_w = img_np.shape[:2]

    # Nasz bbox w formacie xyxy
    x, y, w, h = map(float, row["bbox_parsed"])
    our_box = [x, y, x+w, y+h]

    try:
        # YOLO inference na całym obrazie
        results = yolo_model(image_pil, verbose=False)[0]

        best_iou  = 0
        best_mask = None

        if results.masks is not None:
            for i, yolo_box in enumerate(results.boxes.xyxy.cpu().numpy()):
                iou = compute_iou(our_box, yolo_box)
                if iou > best_iou:
                    best_iou  = iou
                    # Maska YOLO jest w rozmiarze obrazu
                    mask_data = results.masks.data[i].cpu().numpy()
                    # Resize maski do rozmiaru obrazu jeśli potrzeba
                    if mask_data.shape != (img_h, img_w):
                        mask_data = cv2.resize(
                            mask_data, (img_w, img_h),
                            interpolation=cv2.INTER_NEAREST
                        )
                    best_mask = mask_data > 0.5

        if best_mask is not None and best_iou >= iou_threshold:
            # Rozszerz maskę żeby nie obcinać nóg/ogona
            kernel    = np.ones((15, 15), np.uint8)
            best_mask = cv2.dilate(best_mask.astype(np.uint8), kernel).astype(bool)

            # Przyciemnij tło
            masked      = img_np.copy().astype(float)
            masked[~best_mask] = masked[~best_mask] * bg_darken
            masked      = np.clip(masked, 0, 255).astype(np.uint8)
            masked_pil  = Image.fromarray(masked)
        else:
            # Fallback — standardowy crop
            masked_pil = image_pil

        # Crop kwadratowy z paddingiem
        crop = crop_square(masked_pil, row["bbox_parsed"], padding=padding)
        return crop, best_iou

    except Exception as e:
        return crop_square(image_pil, row["bbox_parsed"], padding=padding), 0.0

In [ ]:
# ── Generuj maski dla train2 + test ──
all_df = pd.concat([
    train_df[["row_id", "image_id", "source", "bbox_parsed"]],
    test[["row_id",    "image_id", "source", "bbox_parsed"]],
], ignore_index=True)

print(f"Łącznie instancji do przetworzenia: {len(all_df)}")
print("Szacowany czas: ~10-20 minut na T4\n")

fallback_count = 0
low_iou_count  = 0

for _, row in tqdm(all_df.iterrows(), total=len(all_df), desc="Generowanie masek"):
    out_path = mask_path_for_row(row)

    # Pomiń jeśli już wygenerowane (resume po resecie sesji)
    if os.path.exists(out_path):
        continue

    crop, iou = generate_masked_crop(row, yolo_model)

    if iou == 0.0:
        fallback_count += 1
    elif iou < 0.3:
        low_iou_count += 1

    crop.save(out_path)

print(f"\nGotowe!")
print(f"Fallbacki (YOLO nic nie znalazł): {fallback_count} / {len(all_df)}")
print(f"Niskie IoU (<0.3):                 {low_iou_count} / {len(all_df)}")
print(f"Maski zapisane w: {MASK_DIR}/")

## Wizualizacja — jak YOLO wycina świnie

Porównanie: oryginalny crop vs YOLO masked crop dla każdej klasy.

In [ ]:
def visualize_yolo_crops(df, n_per_class=2):
    """Pokaż po n_per_class przykładów dla każdej klasy."""
    classes = sorted(df["class_id"].unique())
    n_cols  = n_per_class * 2  # oryginał + yolo dla każdej próbki

    fig, axes = plt.subplots(
        len(classes), n_cols,
        figsize=(3.5 * n_cols, 3.5 * len(classes))
    )

    for row_idx, class_id in enumerate(classes):
        samples = df[df["class_id"] == class_id].sample(
            n=min(n_per_class, len(df[df["class_id"] == class_id])),
            random_state=42
        )

        col = 0
        for _, row in samples.iterrows():
            # Oryginalny crop
            orig = crop_square(load_image(row["image_id"], row["source"]), row["bbox_parsed"])
            axes[row_idx, col].imshow(orig)
            axes[row_idx, col].set_title(f"Oryginał\n{CLASS_NAMES[class_id]}", fontsize=8)
            axes[row_idx, col].axis("off")
            col += 1

            # YOLO masked crop
            mp = mask_path_for_row(row)
            if os.path.exists(mp):
                yolo_crop = Image.open(mp).convert("RGB")
            else:
                yolo_crop = orig

            axes[row_idx, col].imshow(yolo_crop)
            axes[row_idx, col].set_title("YOLO masked", fontsize=8)
            axes[row_idx, col].axis("off")
            col += 1

    plt.suptitle("Oryginalny crop  vs  YOLO masked crop", fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.show()


visualize_yolo_crops(train_df, n_per_class=3)

## Dataset — wczytuje gotowe maski z dysku

Brak overhead YOLOv8 podczas treningu — pełna prędkość.

In [ ]:
class AddGaussianNoise:
    def __init__(self, mean=0.0, std=0.05, p=0.3):
        self.mean, self.std, self.p = mean, std, p
    def __call__(self, tensor):
        if torch.rand(1).item() < self.p:
            tensor = torch.clamp(
                tensor + torch.randn_like(tensor) * self.std + self.mean, 0, 1)
        return tensor


train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.0),   # NIE flipujemy — psuje left/right
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.15, hue=0.05),
    transforms.ToTensor(),
    AddGaussianNoise(p=0.2, std=0.03),
    transforms.RandomErasing(p=0.15, scale=(0.01, 0.05), ratio=(0.75, 1.33), value="random"),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
class PigMaskedDataset(Dataset):
    """
    Wczytuje gotowe YOLO-masked cropy z dysku.
    Jeśli plik nie istnieje → fallback na standardowy crop.
    """
    def __init__(self, df, transform=None, is_test=False):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.is_test   = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        mp       = mask_path_for_row(row)

        if os.path.exists(mp):
            crop = Image.open(mp).convert("RGB")
        else:
            # Fallback — standardowy crop
            image = load_image(row["image_id"], row["source"])
            crop  = crop_square(image, row["bbox_parsed"])

        if self.transform:
            crop = self.transform(crop)

        if self.is_test:
            return crop, row["row_id"]

        return crop, int(row["class_id"])

## DataLoaders

In [ ]:
# WeightedRandomSampler
train_targets  = train_df["class_id"].values
class_counts   = np.bincount(train_targets, minlength=len(CLASS_NAMES))
class_weights  = 1.0 / np.maximum(class_counts, 1)
sample_weights = np.array([class_weights[y] for y in train_targets])

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True,
)

print("Class counts:", {CLASS_NAMES[i]: int(c) for i, c in enumerate(class_counts)})

In [ ]:
train_dataset = PigMaskedDataset(train_df, transform=train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=2,    # normalna prędkość — brak YOLO overhead
    pin_memory=True,
)

print(f"Train batches: {len(train_loader)}")

# Sanity check
imgs, lbls = next(iter(train_loader))
print("Batch shape:", imgs.shape)
print("Labels:", lbls[:8].tolist())

## Model — EfficientNet V2 S

In [ ]:
num_classes = len(CLASS_NAMES)

weights = models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
model   = models.efficientnet_v2_s(weights=weights)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)
model = model.to(device)

# Label smoothing — zapobiega overfittingowi
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(f"Parametry: {sum(p.numel() for p in model.parameters()):,}")

## Trening z checkpointami

In [ ]:
def train_one_epoch(model, loader):
    model.train()
    total_loss, all_preds, all_labels = 0, [], []

    for images, labels in tqdm(loader, leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        all_preds.extend(outputs.argmax(1).detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    n = len(loader.dataset)
    return (total_loss/n,
            accuracy_score(all_labels, all_preds),
            f1_score(all_labels, all_preds, average="macro"))

In [ ]:
os.makedirs("checkpoints", exist_ok=True)
history = []

for epoch in range(EPOCHS):
    loss, acc, f1 = train_one_epoch(model, train_loader)
    scheduler.step()

    history.append({"epoch": epoch+1, "loss": loss, "acc": acc, "f1": f1})
    print(f"Epoch {epoch+1}/{EPOCHS} | loss {loss:.4f} | acc {acc:.4f} | f1 {f1:.4f}")

    torch.save({
        "epoch":            epoch + 1,
        "model_state_dict": model.state_dict(),
        "train_f1":         f1,
    }, f"checkpoints/epoch_{epoch+1}.pt")

print("\nTrening zakończony!")

In [ ]:
# Wykres
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs_x = [h["epoch"] for h in history]
ax1.plot(epochs_x, [h["loss"] for h in history], marker="o")
ax1.set_title("Train Loss"); ax1.set_xlabel("Epoch"); ax1.grid(True)
ax2.plot(epochs_x, [h["f1"] for h in history], marker="o", color="green")
ax2.set_title("Train F1 (macro)"); ax2.set_xlabel("Epoch"); ax2.grid(True)
plt.tight_layout(); plt.show()

### Wybierz epokę do submisji

Zwykle epoka 4-5 generalizuje lepiej niż ostatnia (mniej overfittingu).

In [ ]:
BEST_EPOCH = 5

ckpt = torch.load(f"checkpoints/epoch_{BEST_EPOCH}.pt", map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Załadowano epokę {BEST_EPOCH} (train_f1={ckpt['train_f1']:.4f})")

In [ ]:
from google.colab import files
files.download(f"checkpoints/epoch_{BEST_EPOCH}.pt")

## Inference z TTA

Tylko brightness augmentation — bez flipów (flipy psują Lateral_lying_left vs right).

In [ ]:
test_dataset = PigMaskedDataset(test, transform=val_transform, is_test=True)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

print("Test batches:", len(test_loader))

In [ ]:
def predict_with_tta(model, loader, device):
    """
    TTA tylko z brightness — NIE używamy flipów bo psują left/right klasy.
    3 warianty: normalny + jaśniejszy + ciemniejszy
    """
    model.eval()
    all_row_ids, all_probs = [], []

    with torch.no_grad():
        for images, ids in tqdm(loader, desc="TTA Inference"):
            images = images.to(device)

            brighter = torch.clamp(images * 1.15, -3, 3)
            darker   = torch.clamp(images * 0.85, -3, 3)

            p1 = F.softmax(model(images),   dim=1)
            p2 = F.softmax(model(brighter), dim=1)
            p3 = F.softmax(model(darker),   dim=1)

            avg = (p1 + p2 + p3) / 3.0
            all_probs.extend(avg.cpu().numpy())
            all_row_ids.extend(ids)

    return all_row_ids, all_probs


row_ids, probs = predict_with_tta(model, test_loader, device)
predictions = [int(np.argmax(p)) for p in probs]
print("Predykcje gotowe:", len(predictions))

## Submission

In [ ]:
submission = pd.DataFrame({
    "row_id":   row_ids,
    "class_id": predictions,
})

assert len(submission) == len(test), f"Błąd: {len(submission)} != {len(test)}"
assert set(submission["class_id"].unique()).issubset({0,1,2,3,4})

fname = f"T2_efficientnetv2_yolo_ep{BEST_EPOCH}.csv"
submission.to_csv(fname, index=False)

print(f"Zapisano: {fname}")
print(submission["class_id"].value_counts().sort_index())
submission.head()

In [ ]:
from google.colab import files
files.download(fname)